# 02 Preprocessing: add synthetic prices

ABO listings have no price field. This notebook adds **simulated (synthetic) prices** so price filtering (e.g. 'under $50') can be demonstrated. They are not real prices.

Prerequisites: `01_data_exploration.ipynb` saved `listings_en_raw.parquet` to Drive, and `src/preprocess/synthetic_prices.py` is committed to the repo.

In [10]:
import os
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/shoptalk'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        url = f'https://{token}@github.com/anbuaru/shoptalk.git'
    except Exception:
        url = 'https://github.com/anbuaru/shoptalk.git'
    !git clone {url} {REPO}
else:
    !git -C {REPO} pull

%cd /content/shoptalk
!pip install -q -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 12 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 4.56 KiB | 777.00 KiB/s, done.
From https://github.com/anbuaru/shoptalk
   a782a15..37a7fb0  main       -> origin/main
Updating a782a15..37a7fb0
Fast-forward
 preprocess/02_preprocessing.ipynb | 416 ++++++++++++++++++++++++++++++++------
 1 file changed, 355 insertions(+), 61 deletions(-)
/content/shoptalk
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [11]:
import sys
import os
import pandas as pd

sys.path.append(REPO)
assert os.path.exists(f'{REPO}/preprocess/synthetic_prices.py'), 'Commit preprocess/synthetic_prices.py to GitHub first'
from preprocess.synthetic_prices import synthetic_price

PROJECT_DIR = '/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk'
df = pd.read_parquet(f'{PROJECT_DIR}/data/listings_en_raw.parquet')
print(df.shape)
df.head(3)

(147702, 10)


,item_id,name_en,brand_en,color_en,ptype,bullets_en,desc_en,keywords_en,main_image_id,country
0,B06X9STHNG,None,None,None,SHOES,,,,81iZlv3bjpL,NL
1,B07P8ML82R,None,None,None,HARDWARE,,,,619y9YG9cnL,MX
2,B07H9GMYXS,"AmazonBasics PETG 3D Printer Filament, 1.75mm,...",AmazonBasics,Translucent Yellow,MECHANICAL_COMPONENTS,3D printer filament with 1.75mm diameter + / -...,,"3d printer filament, petg printer filament, pe...",81NP7qh2L6L,AE


In [12]:
df['price'] = [synthetic_price(i, t) for i, t in zip(df['item_id'], df['ptype'])]
df['price_currency'] = 'USD'
df['price_is_synthetic'] = True
df[['item_id', 'ptype', 'price', 'price_is_synthetic']].head(10)

,item_id,ptype,price,price_is_synthetic
0,B06X9STHNG,SHOES,54.99,True
1,B07P8ML82R,HARDWARE,44.99,True
2,B07H9GMYXS,MECHANICAL_COMPONENTS,32.99,True
3,B07CTPR73M,SOFA,500.99,True
4,B01MTEI8M6,SHOES,38.99,True
5,B0853X2F4M,CELLULAR_PHONE_CASE,9.99,True
6,B07R91S92W,CELLULAR_PHONE_CASE,9.99,True
7,B0896LJNLH,DRINKING_CUP,81.99,True
8,B0853WVZ6Q,CELLULAR_PHONE_CASE,24.99,True
9,B07R51S57D,FURNITURE_COVER,129.99,True


#Sanity-check the prices

In [13]:
print(df['price'].describe())

med = df.groupby('ptype')['price'].median().sort_values()
print('Cheapest types by median price:')
print(med.head(10))
print('Most expensive types by median price:')
print(med.tail(10))

under50 = df[df['price'] < 50]
print(len(under50), 'items under $50')
print(under50['ptype'].value_counts().head(10))

# Determinism check: running the generator again must give identical prices
again = [synthetic_price(i, t) for i, t in zip(df['item_id'].head(1000), df['ptype'].head(1000))]
assert again == df['price'].head(1000).tolist(), 'Prices are not deterministic'
print('Determinism check passed')

count    147702.000000
mean         43.377239
std          81.652684
min           1.990000
25%          13.990000
50%          23.990000
75%          47.990000
max        2345.990000
Name: price, dtype: float64
Cheapest types by median price:
ptype
AIR_PURIFIER                    10.99
VEHICLE_SAFETY_CAMERA           11.99
GROCERY                         11.99
CARRYING_CASE_OR_BAG            12.99
CE_CARRYING_CASE_OR_BAG         13.99
KICK_SCOOTER                    13.99
MEMORY_READER                   13.99
CELLULAR_PHONE_CASE             13.99
COSMETIC_CASE                   14.99
AMAZON_BOOK_READER_ACCESSORY    14.99
Name: price, dtype: float64
Most expensive types by median price:
ptype
OUTBUILDING           128.99
MECHANICAL_LIGHTER    150.99
TABLE                 184.49
SPORT_TABLE_GAME      204.99
DESK                  204.99
MATTRESS              365.99
BED                   420.99
BED_FRAME             420.99
AIR_MATTRESS          480.99
SOFA                  660.99
Name: pr

In [14]:
out = f'{PROJECT_DIR}/data/listings_en_priced.parquet'
df.to_parquet(out)
print('Saved', out, df.shape)

Saved /content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk/data/listings_en_priced.parquet (147702, 13)


In [15]:
print(df['price'].describe())
print(df.groupby('ptype')['price'].median().sort_values().head(10))
print(df.groupby('ptype')['price'].median().sort_values().tail(10))

# Does the demo query have results?
under50 = df[df['price'] < 50]
print(len(under50), 'items under $50')
print(under50['ptype'].value_counts().head(10))

count    147702.000000
mean         43.377239
std          81.652684
min           1.990000
25%          13.990000
50%          23.990000
75%          47.990000
max        2345.990000
Name: price, dtype: float64
ptype
AIR_PURIFIER                    10.99
VEHICLE_SAFETY_CAMERA           11.99
GROCERY                         11.99
CARRYING_CASE_OR_BAG            12.99
CE_CARRYING_CASE_OR_BAG         13.99
KICK_SCOOTER                    13.99
MEMORY_READER                   13.99
CELLULAR_PHONE_CASE             13.99
COSMETIC_CASE                   14.99
AMAZON_BOOK_READER_ACCESSORY    14.99
Name: price, dtype: float64
ptype
OUTBUILDING           128.99
MECHANICAL_LIGHTER    150.99
TABLE                 184.49
SPORT_TABLE_GAME      204.99
DESK                  204.99
MATTRESS              365.99
BED                   420.99
BED_FRAME             420.99
AIR_MATTRESS          480.99
SOFA                  660.99
Name: price, dtype: float64
112955 items under $50
ptype
CELLULAR_PHONE_CASE  

In [16]:
!git pull

Already up to date.


In [17]:
for t in ['SHIRT', 'BOOT', 'SHOES', 'SANDAL', 'WALLET', 'BAG', 'LAMP', 'SOFA', 'CHAIR', 'RING', 'WATCH', 'COFFEE']:
    print(f'{t:8s}', df['ptype'].str.contains(t, na=False).sum())
print(df['ptype'].value_counts().head(40))

SHIRT    0
BOOT     2009
SHOES    12965
SANDAL   1845
WALLET   176
BAG      1324
LAMP     691
SOFA     1199
CHAIR    2173
RING     3859
WATCH    112
COFFEE   313
ptype
CELLULAR_PHONE_CASE           64853
SHOES                         12965
GROCERY                        6546
HOME                           5264
HOME_BED_AND_BATH              3082
HOME_FURNITURE_AND_DECOR       2255
CHAIR                          2100
BOOT                           2009
SANDAL                         1845
FINERING                       1540
HEALTH_PERSONAL_CARE           1449
FINENECKLACEBRACELETANKLET     1377
ACCESSORY                      1362
SOFA                           1199
OFFICE_PRODUCTS                1152
FINEEARRING                    1137
PET_SUPPLIES                   1064
SPORTING_GOODS                  972
TABLE                           936
HARDWARE_HANDLE                 860
RUG                             817
HANDBAG                         792
LIGHT_BULB                      756
KITC

##There are no shirts at all (SHIRT is 0), so the “red shirt for men” example has to go, and apparel here means shoes, boots, sandals, hats, handbags, and jewelry. Shoes (12,965 items, 9%) and sandals are the apparel strength. The rest of the catalog is mostly home and furniture, which suits “cozy living room” and gift-style queries. Phone cases are still 44% of the data and will be mostly dropped.

##Proposed scope: about 28k products across roughly 30 types
Footwear and accessories: SHOES, BOOT, SANDAL, HANDBAG, HAT, SUITCASE, plus anything with WALLET or WATCH in the type
Jewelry: FINERING, FINEEARRING, EARRING, FINENECKLACEBRACELETANKLET, NECKLACE
Furniture and decor: SOFA, CHAIR, TABLE, RUG, LAMP, LIGHT_FIXTURE, OTTOMAN, STOOL_SEATING, WALL_ART, HOME_FURNITURE_AND_DECOR
Home and gifts: HOME_BED_AND_BATH, FLAT_SHEET, DRINKING_CUP, KITCHEN, PET_SUPPLIES, plus anything with COFFEE in the type
Distractors: 1,000 phone cases, so retrieval has to ignore them

Each type is capped at 2,000 items so shoes can’t dominate. That size is also small enough to caption in about an hour or two on a Colab T4.

In [18]:
KEEP = ['SHOES', 'BOOT', 'SANDAL', 'HANDBAG', 'HAT', 'SUITCASE',
        'FINERING', 'FINEEARRING', 'EARRING', 'FINENECKLACEBRACELETANKLET', 'NECKLACE',
        'SOFA', 'CHAIR', 'TABLE', 'RUG', 'LAMP', 'LIGHT_FIXTURE', 'OTTOMAN',
        'STOOL_SEATING', 'WALL_ART', 'HOME_FURNITURE_AND_DECOR',
        'HOME_BED_AND_BATH', 'FLAT_SHEET', 'DRINKING_CUP', 'KITCHEN', 'PET_SUPPLIES']
CAP = 2000
N_DISTRACTORS = 1000

# 1. Quality filter: needs an English name and some descriptive text
has_text = (df['bullets_en'].str.len() > 0) | (df['desc_en'].str.len() > 0) | (df['keywords_en'].str.len() > 0)
ok = df[df['name_en'].notna() & has_text & df['main_image_id'].notna()]

# 2. Keep chosen types, plus the small wallet/watch/coffee types
keep_mask = ok['ptype'].isin(KEEP) | ok['ptype'].str.contains('WALLET|WATCH|COFFEE', na=False)
sub = ok[keep_mask]

# 3. Cap each type (shuffle first so the cap is a random sample)
sub = sub.sample(frac=1, random_state=0).groupby('ptype').head(CAP)

# 4. Add phone cases as distractors
distractors = ok[ok['ptype'] == 'CELLULAR_PHONE_CASE'].sample(N_DISTRACTORS, random_state=0)
scoped = pd.concat([sub, distractors]).reset_index(drop=True)

print(scoped.shape)
print(scoped['ptype'].value_counts().head(40))
print('Types kept:', scoped['ptype'].nunique())

(18938, 13)
ptype
SHOES                         2000
CHAIR                         1557
HOME_FURNITURE_AND_DECOR      1232
HOME_BED_AND_BATH             1077
CELLULAR_PHONE_CASE           1000
BOOT                           972
SOFA                           930
FINENECKLACEBRACELETANKLET     904
FINERING                       838
RUG                            816
SANDAL                         807
FINEEARRING                    731
TABLE                          720
HANDBAG                        676
WALL_ART                       598
PET_SUPPLIES                   496
LIGHT_FIXTURE                  466
KITCHEN                        451
LAMP                           441
OTTOMAN                        347
STOOL_SEATING                  345
NECKLACE                       272
SUITCASE                       230
COFFEE                         228
EARRING                        203
DRINKING_CUP                   154
FLAT_SHEET                     142
HAT                            117
WA

In [19]:
out = f'{PROJECT_DIR}/data/listings_en_scoped.parquet'
scoped.to_parquet(out)
print('Saved', out)

Saved /content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk/data/listings_en_scoped.parquet


In [21]:
df.shape

(147702, 13)

In [24]:
df.dtypes

,0
item_id,object
name_en,object
brand_en,object
color_en,object
ptype,object
bullets_en,object
desc_en,object
keywords_en,object
main_image_id,object
country,object


One small bug: SWATCH (67 rows) slipped in, because 'WATCH' matched as a substring. Drop it and keep the real watches (14 items, which is too few for a watch query, so I’d skip watch examples).

Cells to add next: clean the data and build the text field

This is the text each product will be embedded from. Price is deliberately left out of it.

In [25]:
import re, html

# Remove the SWATCH accident
scoped = scoped[scoped['ptype'] != 'SWATCH'].reset_index(drop=True)

def clean(s):
    if not isinstance(s, str):
        return ''
    s = html.unescape(s)
    s = re.sub(r'<[^>]+>', ' ', s)          # strip HTML tags
    s = re.sub(r'\s+', ' ', s).strip()
    return s

for c in ['name_en', 'brand_en', 'color_en', 'bullets_en', 'desc_en', 'keywords_en']:
    scoped[c] = scoped[c].apply(clean)

def build_text(r):
    parts = [r['name_en']]
    if r['brand_en']:  parts.append(f"Brand: {r['brand_en']}.")
    parts.append(f"Type: {r['ptype'].replace('_', ' ').lower()}.")
    if r['color_en']:  parts.append(f"Color: {r['color_en']}.")
    if r['bullets_en']: parts.append(r['bullets_en'])
    if r['desc_en']:    parts.append(r['desc_en'])
    if r['keywords_en']: parts.append(f"Keywords: {r['keywords_en']}.")
    return ' '.join(parts)[:1500]            # key facts first, then trim long tails

scoped['text'] = scoped.apply(build_text, axis=1)

# Drop exact duplicate texts (variants with identical listings)
before = len(scoped)
scoped = scoped.drop_duplicates(subset='text').reset_index(drop=True)
print('Dropped duplicates:', before - len(scoped), '| rows now:', len(scoped))

# Length check (embedding models read roughly 256-512 tokens)
print(scoped['text'].str.split().str.len().describe())

# Read a few finished texts
for t in scoped.sample(3, random_state=1)['text']:
    print(t[:600], '\n---')

Dropped duplicates: 285 | rows now: 18586
count    18586.000000
mean       135.491176
std         75.432906
min          8.000000
25%         65.000000
50%        132.000000
75%        211.000000
max        299.000000
Name: text, dtype: float64
Amazon Brand – Rivet Edgewest Low Back Modern Accent Chair, 34"W, Slate Brown Brand: Rivet. Type: chair. Color: Slate Brown. Features such as track arms, box-edge framing and metal legs make this collection look sharp in modern or industrial settings. Back and seat cushions are reversible, and the collection offers multiple options to accommodate family and friends. 34"W x 33"D x 32"H; Seat Height: 19"H ; Seat Depth: 23"D; Seat Back Height: 25"H; Arm Height: 25"H; Leg Height: 5.25"H Polyester fabric on hardwood pine frame; metal legs with bronze finish Crisp, clean lines and metal legs are hal 
---
Amazon Basics Suitcase Carrying Case with Expandable Capacity and Built-in TSA Lock, Soft Side, Elastic Handle - blue Type: suitcase. Color: bule. Pr

In [26]:
out = f'{PROJECT_DIR}/data/listings_en_clean.parquet'
scoped.to_parquet(out)
print('Saved', out, scoped.shape)

Saved /content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk/data/listings_en_clean.parquet (18586, 14)
